# Getting started with OpenNDM

A complete calculation from nothing, in five objects:

| Object | What it holds |
|---|---|
| `XSLibrary` | group constants, one row per composition |
| `Geometry`  | which composition sits where, and the boundaries |
| `Settings`  | kernel choice and convergence criteria |
| `Model`     | the three above, plus a persistent solver |
| `Result`    | eigenvalue, flux, power, iteration history |

Nothing here needs OpenMC. The solver is independent of it; only
`openndm.gc` requires it.

In [ ]:
import numpy as np
import openndm

print("openndm", openndm.__version__)

## 1. Cross sections

Two compositions, two energy groups: a fuel and a reflector. Group 1 is the
fast group, following OpenMC's convention that group 1 sits at the highest
energy.

The scattering matrix is indexed `scatter[from_group][to_group]`, so the
top-right entry is down-scatter out of the fast group. Getting that
orientation wrong is a classic error and it is not subtle in its effects: the
removal cross section comes out wrong for every group.

In [ ]:
lib = openndm.XSLibrary(n_groups=2, n_compositions=2)

lib.set_composition(
    0,                                     # fuel
    D=[1.5, 0.4],
    absorption=[0.010, 0.085],
    nu_fission=[0.0, 0.135],
    kappa_fission=[0.0, 0.135],            # needed for a power distribution
    chi=[1.0, 0.0],                        # all fission neutrons born fast
    scatter=[[0.0, 0.020],                 # fast -> thermal
             [0.0, 0.000]],
)

lib.set_composition(
    1,                                     # reflector: no fission
    D=[2.0, 0.3],
    absorption=[0.000, 0.010],
    scatter=[[0.0, 0.040],
             [0.0, 0.000]],
)

warnings_raised = lib.finalize()
print("validation warnings:", warnings_raised or "none")
print(lib)

`finalize()` validates and caches. It **raises** on data that cannot be
physical — a non-positive diffusion coefficient, a negative absorption cross
section, a fission spectrum that does not sum to one — and **warns** on data
that is merely suspicious, such as negative scattering transfers, which Monte
Carlo noise produces routinely, or a fissile composition with no
`kappa_fission`, whose only symptom would be a silently zero power
distribution.

## 2. Geometry

A composition map with shape `(nz, ny, nx)`. `openndm.INACTIVE` marks a
position outside the core.

In [ ]:
core = np.zeros((10, 9, 9), dtype=int)
core[:, 8, :] = 1          # radial reflector on the +y edge
core[:, :, 8] = 1          # and the +x edge
core[0] = core[-1] = 1     # axial reflectors
core[:, 7, 7] = openndm.INACTIVE   # a corner outside the core

geom = openndm.Geometry.from_lattice(
    core,
    pitch=(20.0, 20.0, 20.0),
    boundaries={
        "x_min": "reflective",   # quarter-core symmetry planes
        "y_min": "reflective",
        "x_max": "zero_flux",
        "y_max": "zero_flux",
        "z_min": "vacuum",
        "z_max": "vacuum",
    },
    outside="zero_flux",         # faces looking at an INACTIVE position
)
print(geom)

`outside` is deliberately separate from `boundaries`. On a quarter-core map
the mesh edges carry the *symmetry* conditions, while a face looking at an
out-of-core position is a real outer boundary. Conflating the two reflects
neutrons back into the core from outside it, which inflates `k_eff` by
hundreds of pcm with no other symptom.

## 3. Solve

In [ ]:
settings = openndm.Settings(kernel="sanm", verbosity=0)
model = openndm.Model(geom, lib, settings)
result = model.solve()
result

In [ ]:
print(f"k_eff             {result.k_eff:.6f}")
print(f"converged         {result.converged} in {result.outer_iterations} outers")
print(f"runtime           {result.runtime * 1e3:.1f} ms")
print(f"flux              {result.flux.shape}  (n_nodes, n_groups)")
print(f"F_q  (peak node)  {result.f_q:.4f}")
print(f"F_dH (peak radial){result.f_dh:.4f}")

## 4. Compare the kernels

Three are available and the choice is made at run time. FDM is plain finite
difference; NEM and SANM solve a transverse-integrated nodal problem in each
node and correct the coupling coefficients accordingly.

In [ ]:
for kernel in ("fdm", "nem", "sanm"):
    r = model.solve(kernel=kernel)
    print(f"{kernel:5s} k_eff = {r.k_eff:.6f}   "
          f"{r.outer_iterations:3d} outers   {r.runtime * 1e3:6.1f} ms")

On a 20 cm mesh the three disagree by hundreds of pcm; that gap *is* the
nodal correction. Refine the mesh and they converge together — `subdivide`
splits each lattice cell without changing the material layout.

In [ ]:
for sub in (1, 2, 4):
    g = openndm.Geometry.from_lattice(
        core, pitch=(20.0, 20.0, 20.0), subdivide=(sub, sub, 1),
        boundaries={"x_min": "reflective", "y_min": "reflective",
                    "x_max": "zero_flux", "y_max": "zero_flux",
                    "z_min": "vacuum", "z_max": "vacuum"},
        outside="zero_flux")
    m = openndm.Model(g, lib, settings)
    ks = {k: m.solve(kernel=k).k_eff for k in ("fdm", "nem", "sanm")}
    spread = 1e5 * (max(ks.values()) - min(ks.values()))
    print(f"{sub} node(s)/assembly ({g.n_nodes:5d} nodes):  "
          + "  ".join(f"{k.upper()} {v:.6f}" for k, v in ks.items())
          + f"   spread {spread:6.1f} pcm")

## 5. Results

Node-ordered arrays can be scattered back onto the lattice with
`geometry.expand`, and there are ready-made collapses for the usual
quantities.

In [ ]:
radial = result.radial_power()      # (ny, nx), volume weighted
axial = result.axial_power()        # (nz,)
print("radial power map, mean 1.0 over powered positions:\n")
for row in radial[::-1]:
    print("  " + " ".join("  .  " if v == 0 else f"{v:5.3f}" for v in row))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

masked = np.ma.masked_where(radial <= 0, radial)
im = axes[0].imshow(masked, origin="lower", cmap="viridis")
fig.colorbar(im, ax=axes[0], label="relative power")
axes[0].set_title("radial power")

axes[1].step(axial, np.arange(axial.size), where="mid")
axes[1].set_title("axial power")
axes[1].set_xlabel("relative power")
axes[1].set_ylabel("plane")
axes[1].grid(alpha=0.3)

history = result.history
axes[2].semilogy(history["outer"], np.abs(history["k_change"]), label=r"$|\Delta k|$")
axes[2].semilogy(history["outer"], history["source_change"], label="fission source")
axes[2].set_title("convergence")
axes[2].set_xlabel("outer iteration")
axes[2].legend()
axes[2].grid(alpha=0.3, which="both")
fig.tight_layout()

`openndm.plots` wraps these: `plot_radial`, `plot_axial` and
`plot_convergence` all take an optional `ax` and return it.

## 6. Check the answer

Two habits worth keeping. First, every node should conserve neutrons; the
residual is normalised, so it is comparable between nodes.

In [ ]:
residual = np.abs(model.neutron_balance())
print(f"worst node-wise neutron balance residual: {residual.max():.2e}")

Second, refine the mesh and see whether the answer moves. A nodal method
should barely move, which is what distinguishes it from finite difference; if
it moves a lot, the nodal correction is not doing its job.

## 7. Writing results out

`statepoint.h5` follows OpenMC's conventions, so a reader fluent in
`openmc.StatePoint` needs no new habits.

In [ ]:
import tempfile
from pathlib import Path

path = Path(tempfile.mkdtemp()) / "statepoint.h5"
openndm.write_statepoint(path, result, model, extra={"case": "getting started"})

with openndm.StatePoint(path) as sp:
    print(f"k_eff {sp.k_eff:.6f}, kernel {sp.kernel!r}, "
          f"written by openndm {sp.version}")
    print(f"radial power shape {sp.radial_power.shape}")

## Next

- **Notebook 2** builds the library from an OpenMC lattice calculation.
- **Notebook 3** runs the IAEA-2D benchmark.
- **Notebook 4** covers branch-parameterised libraries and the critical boron
  search.
- `docs/user-guide.md` is the reference for everything above.